In [1]:
import requests
import re
import json
import pandas as pd
import time
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import os

In [2]:
# 1. Obtener el HTML de la página del ranking
load_dotenv()
url = os.getenv("TARGET_URL_RESERVES")
BASE_URL = os.getenv("BASE_URL")

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
response.raise_for_status()  # Lanza error si la petición falla

In [3]:
# 2. Parsear HTML con BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

In [4]:
# 3. Buscar la tabla principal y extraer los links
table = soup.find("table")
country_links = []


if table:
    rows = table.find_all("tr")
    for row in rows:
        cell = row.find("td")
        if cell and cell.a:
            relative_link = cell.a["href"]
            full_url = f"{BASE_URL}{relative_link}"
            country_links.append(full_url)

# Convertir la lista en DataFrame
df_links = pd.DataFrame(country_links, columns=["url"])

# Guardar como archivo Parquet
df_links.to_parquet("../../data/country_reserves_links.parquet", index=False)

In [5]:
# 4. Mostrar resultado
print("Links encontrados:", df_links.size)

Links encontrados: 162


In [6]:
# configuracion necesaria para descargar datos

master_df = pd.DataFrame()

pattern = r'arrayToDataTable\(\s*(\[\[.*?\]\])\s*\);'

sleep_duration = 5

In [7]:
# Cargar los países válidos desde el archivo parquet
df_countries = pd.read_parquet("../../data/countries_with_hpi.parquet")
valid_countries = set(df_countries["country"].unique())

In [8]:
# DataFrames de salida
dfs_quarters = []
dfs_months = []

start_time = time.time()
for url in country_links:
    country = urlparse(url).path.strip("/").split("/")[0]

    if country not in valid_countries:
        print(f"⏭️ País {country} no está en la lista, se omite.")
        continue

    try:
        response = requests.get(url, headers=headers)
        html = response.text

        match = re.search(pattern, html, re.DOTALL)
        if match:
            data_str = match.group(1)
            data = json.loads(data_str)

            df = pd.DataFrame(data[1:], columns=data[0])
            df.columns = ["Period", "Value"]
            df["Country"] = country

            # --- separar según formato de Period ---
            if df["Period"].str.contains(r"Q\d").any():
                df_pivot = df.pivot(index="Period", columns="Country", values="Value")
                dfs_quarters.append(df_pivot)
                print(f"✅ Datos trimestrales extraídos para {country}")
            else:
                df_pivot = df.pivot(index="Period", columns="Country", values="Value")
                dfs_months.append(df_pivot)
                print(f"✅ Datos mensuales extraídos para {country}")
        else:
            print(f"⚠️ No se encontraron datos en {url}")
    except Exception as e:
        print(f"❌ Error procesando {url}: {e}")
    
    time.sleep(sleep_duration)

# unir cada grupo por separado
df_quarters = pd.concat(dfs_quarters, axis=1) if dfs_quarters else pd.DataFrame()
df_months = pd.concat(dfs_months, axis=1) if dfs_months else pd.DataFrame()

end_time = time.time()
total_sleep_time = len(valid_countries) * sleep_duration
effective_time = end_time - start_time - total_sleep_time

print(f"⏳ Tiempo total (incluyendo esperas): {end_time - start_time:.2f} s")
print(f"😴 Tiempo en espera: {total_sleep_time:.2f} s")
print(f"⚡ Tiempo efectivo de ejecución: {effective_time:.2f} s")

⏭️ País Albania no está en la lista, se omite.
⏭️ País Algeria no está en la lista, se omite.
⏭️ País Angola no está en la lista, se omite.
⏭️ País Argentina no está en la lista, se omite.
⏭️ País Armenia no está en la lista, se omite.
⏭️ País Aruba no está en la lista, se omite.
✅ Datos mensuales extraídos para Australia
✅ Datos mensuales extraídos para Austria
⏭️ País Azerbaijan no está en la lista, se omite.
⏭️ País Bahamas no está en la lista, se omite.
⏭️ País Bahrain no está en la lista, se omite.
⏭️ País Bangladesh no está en la lista, se omite.
⏭️ País Barbados no está en la lista, se omite.
⏭️ País Belarus no está en la lista, se omite.
✅ Datos mensuales extraídos para Belgium
⏭️ País Belize no está en la lista, se omite.
⏭️ País Benin no está en la lista, se omite.
⏭️ País Bhutan no está en la lista, se omite.
⏭️ País Bolivia no está en la lista, se omite.
⏭️ País Bosnia-and-Herzegovina no está en la lista, se omite.
⏭️ País Botswana no está en la lista, se omite.
✅ Datos men

In [9]:
# resultados finales
print("\n📊 DataFrame trimestral")
df_quarters.head()


📊 DataFrame trimestral


""


In [10]:
print("\n📊 DataFrame mensual")
df_months.head()


📊 DataFrame mensual


Country,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,...,Slovenia,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom
Period,,,,,,,,,,,,,,,,,,,,,
Apr 1970,1.34,NaN,NaN,0.94,NaN,3.60,0.39,NaN,0.30,NaN,...,NaN,NaN,0.56,NaN,NaN,NaN,NaN,NaN,16.92,NaN
Apr 1971,2.01,NaN,NaN,1.38,NaN,4.85,0.32,NaN,0.23,NaN,...,NaN,NaN,0.57,NaN,NaN,NaN,NaN,NaN,14.31,NaN
Apr 1972,3.39,NaN,NaN,2.21,NaN,5.75,0.17,NaN,0.25,NaN,...,NaN,NaN,0.38,NaN,NaN,NaN,NaN,NaN,12.47,NaN
Apr 1973,4.22,NaN,NaN,5.18,NaN,5.97,0.13,NaN,0.45,NaN,...,NaN,NaN,0.54,NaN,NaN,NaN,NaN,NaN,13.17,NaN
Apr 1974,3.69,NaN,NaN,6.52,NaN,6.10,0.38,NaN,0.57,NaN,...,NaN,NaN,0.69,NaN,NaN,NaN,NaN,NaN,14.64,NaN


In [11]:
df_months.tail()

Country,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,...,Slovenia,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom
Period,,,,,,,,,,,,,,,,,,,,,
Sep 1966,NaN,NaN,NaN,0.36,NaN,2.75,0.15,NaN,0.12,NaN,...,NaN,NaN,0.20,NaN,NaN,NaN,NaN,NaN,14.88,NaN
Sep 1967,NaN,NaN,NaN,0.21,NaN,2.66,0.10,NaN,0.14,NaN,...,NaN,NaN,0.30,NaN,NaN,NaN,NaN,NaN,14.65,NaN
Sep 1968,NaN,NaN,NaN,0.26,NaN,2.73,0.17,NaN,0.18,NaN,...,NaN,NaN,0.36,NaN,NaN,NaN,NaN,NaN,14.63,NaN
Jul 2025,NaN,NaN,NaN,NaN,84.63,126.57,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Jan 1962,NaN,NaN,NaN,NaN,NaN,NaN,0.03,NaN,0.15,NaN,...,NaN,NaN,0.21,NaN,NaN,NaN,NaN,NaN,18.65,NaN


In [12]:
# obtain paraguay url
template_url = country_links[0]
country = "Paraguay"

# Separar por "/"
parts = template_url.strip("/").split("/")

# Reemplazar el país (posición 3: después del dominio)

parts[3] = country

# Reconstruir la URL
custom_url = "/".join(parts) + "/"

In [13]:
# URL específica para Paraguay
response = requests.get(custom_url, headers=headers)
html = response.text

# Buscar el bloque JSON dentro del HTML
match = re.search(pattern, html, re.DOTALL)
if match:
    data_str = match.group(1)
    data = json.loads(data_str)

    # Crear DataFrame con los datos
    df = pd.DataFrame(data[1:], columns=data[0])
    df.columns = ["Period", "Value"]
    df["Country"] = "Paraguay"  # Asignar explícitamente

    # Pivotear para formato ancho (opcional)
    df_paraguay = df.pivot(index="Period", columns="Country", values="Value")

    print("✅ Datos de Paraguay extraídos correctamente")
else:
    print("⚠️ No se encontraron datos en el HTML para Paraguay")

✅ Datos de Paraguay extraídos correctamente


In [14]:
df_paraguay.tail()

Country,Paraguay
Period,
Sep 2020,8.95
Sep 2021,9.75
Sep 2022,9.44
Sep 2023,9.99
Sep 2024,9.88


In [15]:
# Unir ambos DataFrames por el índice 'Period', preservando todos los datos
new_month_df = pd.merge(df_months, df_paraguay, how="outer", left_index=True, right_index=True)
new_month_df.head()

Country,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay
Period,,,,,,,,,,,,,,,,,,,,,
Apr 1960,NaN,NaN,NaN,0.42,NaN,2.00,0.09,NaN,0.22,NaN,...,NaN,0.14,NaN,NaN,NaN,NaN,NaN,21.28,NaN,0.0
Apr 1961,NaN,NaN,NaN,0.33,NaN,2.12,0.09,NaN,0.18,NaN,...,NaN,0.17,NaN,NaN,NaN,NaN,NaN,19.17,NaN,0.0
Apr 1962,NaN,NaN,NaN,0.42,NaN,1.93,0.01,NaN,0.16,NaN,...,NaN,0.21,NaN,NaN,NaN,NaN,NaN,18.19,NaN,0.0
Apr 1963,NaN,NaN,NaN,0.34,NaN,2.62,0.08,NaN,0.13,NaN,...,NaN,0.13,NaN,NaN,NaN,NaN,NaN,17.14,NaN,0.0
Apr 1964,NaN,NaN,NaN,0.08,NaN,2.52,0.07,NaN,0.16,NaN,...,NaN,0.13,NaN,NaN,NaN,NaN,NaN,16.90,NaN,0.0


In [16]:
def split_period_index(df):
    """
    Recibe un DataFrame con índice 'Period' (ej: 'Apr 1960' o 'Q1 1990')
    Devuelve el mismo DataFrame con columnas 'Year' y 'Month' o 'Quarter'
    """
    df = df.copy().reset_index()  # sacar Period del índice
    df["Year"] = None
    df["Month"] = None
    df["Quarter"] = None

    for i, period in enumerate(df["Period"]):
        parts = period.split()

        if len(parts) == 2:
            first, year = parts
            df.at[i, "Year"] = int(year)

            if "Q" in first:  # Formato trimestral
                df.at[i, "Quarter"] = first
            else:  # Formato mensual
                df.at[i, "Month"] = first

    return df.set_index("Period")  # opcional: devolver con Period otra vez de índice

In [17]:
def month_to_quarter(df):
    """
    Takes a DataFrame with 'Month' and 'Quarter' columns.
    If 'Month' exists, it assigns the corresponding quarter to 'Quarter'.
    """
    df = df.copy()

    # Dictionary for mapping months → quarters
    month_to_q = {
        "Jan": 1, "Feb": 1, "Mar": 1,
        "Apr": 2, "May": 2, "June": 2,
        "Jul": 3, "Aug": 3, "Sep": 3,
        "Oct": 4, "Nov": 4, "Dec": 4,
    }

    # Update quarter only where Month exists
    df["Quarter"] = df.apply(
        lambda row: month_to_q.get(row["Month"], row["Quarter"]),
        axis=1
    )

    return df

In [18]:
def year_to_int(df):
    """
    Convierte la columna 'Year' a enteros.
    Si hay valores no numéricos, los convierte en NaN.
    """
    df = df.copy()
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
    return df

In [19]:
new_month_df = split_period_index(new_month_df)
new_month_df = month_to_quarter(new_month_df)
new_month_df = year_to_int(new_month_df)
new_month_df.dtypes

Country
Australia         float64
Austria           float64
Belgium           float64
Brazil            float64
Bulgaria          float64
                   ...   
United-Kingdom    float64
Paraguay          float64
Year                Int64
Month              object
Quarter             int64
Length: 61, dtype: object

In [20]:
new_month_df.head()

Country,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,...,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay,Year,Month,Quarter
Period,,,,,,,,,,,,,,,,,,,,,
Apr 1960,NaN,NaN,NaN,0.42,NaN,2.00,0.09,NaN,0.22,NaN,...,NaN,NaN,NaN,NaN,21.28,NaN,0.0,1960,Apr,2
Apr 1961,NaN,NaN,NaN,0.33,NaN,2.12,0.09,NaN,0.18,NaN,...,NaN,NaN,NaN,NaN,19.17,NaN,0.0,1961,Apr,2
Apr 1962,NaN,NaN,NaN,0.42,NaN,1.93,0.01,NaN,0.16,NaN,...,NaN,NaN,NaN,NaN,18.19,NaN,0.0,1962,Apr,2
Apr 1963,NaN,NaN,NaN,0.34,NaN,2.62,0.08,NaN,0.13,NaN,...,NaN,NaN,NaN,NaN,17.14,NaN,0.0,1963,Apr,2
Apr 1964,NaN,NaN,NaN,0.08,NaN,2.52,0.07,NaN,0.16,NaN,...,NaN,NaN,NaN,NaN,16.90,NaN,0.0,1964,Apr,2


In [21]:
new_month_df.tail()

Country,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,...,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay,Year,Month,Quarter
Period,,,,,,,,,,,,,,,,,,,,,
Sep 2020,61.56,24.39,27.52,356.61,60.78,90.16,37.82,3281.21,56.99,18.31,...,517.48,938.00,7949.35,79.68,138.96,208.20,8.95,2020,Sep,3
Sep 2021,82.14,28.58,35.26,368.89,63.49,106.27,53.31,3373.02,58.73,24.35,...,546.08,1007.56,8293.60,121.26,250.08,229.94,9.75,2021,Sep,3
Sep 2022,83.58,32.28,39.68,327.58,72.27,102.69,37.78,3193.58,56.34,26.26,...,681.21,872.75,7561.94,107.04,230.15,192.22,9.44,2022,Sep,3
Sep 2023,90.41,28.41,36.48,340.32,72.84,112.45,41.40,3308.47,57.60,2.85,...,634.48,747.86,7736.51,122.20,240.99,203.29,9.99,2023,Sep,3
Sep 2024,93.97,32.78,38.21,372.02,81.70,128.07,45.63,3571.80,63.61,2.93,...,663.60,802.29,7887.67,152.07,250.62,226.85,9.88,2024,Sep,3


In [22]:
# Guardar como archivo Parquet
new_month_df.to_parquet("../../data/foreign_exchange_reserves_values.parquet", index=False)